# Data Extraction and Loading

In this notebook, we retrieve the main datasources :
- Financial News: High-quality English articles (Source: GDELT Project) 
- Social Media: Scrapped Telegram Channels messages (Source: [Telegram API](https://my.telegram.org))
- Market Prices: Daily NASDAQ data (Source: Yahoo Finance)

### Libraries

In [1]:
import yfinance as yf
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))
from src.extract_data import *

### Extraction

#### Financial News: High-quality English articles (Source: GDELT Project) 

We extracted high-quality financial news by querying the GDELT Project database via Google BigQuery, focusing on reputable English-language sources over the period January to March 2026. A randomized daily partitioning strategy was implemented to ensure full temporal density and continuity across the three-month window. Full headlines and article bodies were then scraped using a dedicated content extraction pipeline, with automated filtering to remove low-quality or short entries. 

In [ ]:
PATH_NEWS = "../data/raw/raw_news_urls.csv"  # gdelt data
OUTPUT_NEWS = "../data/raw/raw_news_2026.csv"

In [3]:
# Dataset audit
df_raw = check_density(PATH_NEWS)

Total records: 844
Coverage: 90 / 90 days
Gaps found: 0 days


In [5]:
scrape_content(df_raw, OUTPUT_NEWS)

Starting extraction process...


100%|██████████| 844/844 [17:48<00:00,  1.27s/it]


Complete: 736 valid articles extracted.


,date,headline,body,url,source
0,2026-01-01,Investors are betting these stocks will reap c...,Although stocks closed lower on the last tradi...,https://www.cnbc.com/2025/12/31/investors-are-...,cnbc.com
1,2026-01-01,End of an era: Warren Buffett serves last day ...,In this article BRK.B\n\nBRK.A Follow your fav...,https://www.cnbc.com/2026/01/01/warren-buffett...,cnbc.com
2,2026-01-01,A 5 million percent return in 60 years leaves ...,Warren Buffett and Greg Abel walkthrough the B...,https://www.cnbc.com/2026/01/01/warren-buffett...,cnbc.com
3,2026-01-01,"Dust to data centers: The year AI tech giants,...","The Stargate AI data center in Abilene, Texas,...",https://www.cnbc.com/2025/12/31/ai-data-center...,cnbc.com
4,2026-01-01,2026 will bring more GLP-1 weight loss pills —...,The ripple is about to become a wave. Right no...,https://www.cnbc.com/2026/01/01/2026-will-brin...,cnbc.com
...,...,...,...,...,...
731,2026-03-31,Here are Tuesday's biggest analyst calls: Nvid...,Here are the biggest calls on Wall Street on T...,https://www.cnbc.com/2026/03/31/tuesday-top-wa...,cnbc.com
732,2026-03-31,Evercore ISI predicts 'inflection point' is da...,"A week of ""maximum uncertainty"" should deliver...",https://www.cnbc.com/2026/03/30/buy-stocks-if-...,cnbc.com
733,2026-03-31,South Korea proposes extra $17 billion budget ...,Workers install solar power infrastructure at ...,https://www.cnbc.com/2026/03/31/south-korea-bu...,cnbc.com
734,2026-03-31,Snap climbs 14% as activist Irenic suggests ch...,Snap CEO Evan Spiegel onstage during the Snap ...,https://www.cnbc.com/2026/03/31/snap-stock-act...,cnbc.com


#### Social Media: Crowd sentiment and engagement 

We collected high-frequency financial messages from multiple Telegram channels using a custom scraping pipeline built on the Telethon API, focusing on NASDAQ-related keywords and cashtags over the January–March 2026 period. A regex-based filtering strategy was applied to retain only posts with major tech stocks and market sentiment signals. 

Text data was preprocessed by removing emojis and non-ASCII characters to ensure consistency for downstream NLP tasks. Each message was enriched with metadata such as timestamps, channel source, and view counts. This results in a structured social signal dataset capturing real-time retail sentiment aligned with NASDAQ market dynamics.

In [ ]:
OUTPUT_TWEETS = "../data/raw/raw_tweets_2026.csv"

df = scrape_telegram_nasdaq(
    API_ID   = 34806017,
    API_HASH = "d8973e5b317e0808635e54d81e543379",
    PHONE    = "+33750013431",
    output_path=OUTPUT_TWEETS
)

Signed in successfully as Roland Dutauziet; remember to not break the ToS or you will risk an account ban!
Error with nas100masters: No user has "nas100masters" as username
Error with investing_com: Nobody is using this username, or the username is unacceptable. If the latter, it must match r"[a-zA-Z][\w\d]{3,30}[a-zA-Z\d]" (caused by ResolveUsernameRequest)
Error with marketsignals: No user has "marketsignals" as username
Error with stocks_0: Nobody is using this username, or the username is unacceptable. If the latter, it must match r"[a-zA-Z][\w\d]{3,30}[a-zA-Z\d]" (caused by ResolveUsernameRequest)
Collected 3720 messages
Days covered: 90
Saved to: ../data/processed/raw_tweets_2026.csv


#### Market Prices: Daily NASDAQ data (Source: Yahoo Finance)

In [5]:
import yfinance as yf

#  NASDAQ data (^IXIC)
nasdaq = yf.download("^IXIC", start="2025-12-31", end="2026-04-01")
# Delta_d = (Price_d - Price_d-1) / Price_d-1
nasdaq["returns"] = nasdaq["Close"].pct_change()
nasdaq.to_csv("../data/raw/raw_nasdaq_2026.csv")

print(f"Market data saved: {len(nasdaq)} trading days retrieved.")

[*********************100%***********************]  1 of 1 completed

Market data saved: 62 trading days retrieved.
